# Amazon ML Challenge 2026 — Colab Pro End-to-End Pipeline (v3)

**100% Standalone (No Google Drive Required).**

This notebook runs the complete **Business Entity Resolution** pipeline directly on Colab's local storage:
1. **Environment Setup** (Installs Polars, RapidFuzz, AnyASCII, LightGBM)
2. **Self-Extracting Code** (All updated pipeline code is embedded directly inside this notebook)
3. **Dataset Staging** (Automatically detects your uploaded dataset in `/content/`)
4. **Data Normalization** (`prepare.py` with French `cedex` / `bp` cleaning)
5. **Candidate Blocking on Train** (`run_blocking.py` with 98.2% recall blocker)
6. **Leave-One-Country-Out (LOCO) Proxy Test** (`train.py --holdout-country India`)
7. **Candidate Blocking on Test** (`run_blocking.py --split test`)
8. **Full v3 Training** (`train.py` on full train set)
9. **Test Inference & Decision Prediction** (`predict.py`)
10. **Official Submission Validation** (`validate_submission.py`)
11. **One-Click Download** of `matching_results.tsv` for upload to Unstop.

### Cell 1: Check Runtime & Hardware Specs

In [ ]:
!nvidia-smi

import psutil
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"Available System RAM: {ram_gb:.1f} GB")

### Cell 2: Install Required Dependencies

In [ ]:
!pip install -q polars rapidfuzz anyascii lightgbm pyarrow scikit-learn tqdm

### Cell 3: Self-Extract Pipeline Source Code
Extracts all pipeline source files into `/content/code/src/` automatically from the embedded archive.

In [ ]:
import os, base64, io, tarfile

os.makedirs('/content/code', exist_ok=True)
os.makedirs('/content/work', exist_ok=True)
os.makedirs('/content/output', exist_ok=True)
os.makedirs('/content/dataset', exist_ok=True)

# Check if user uploaded a separate code_bundle.tar.gz, otherwise unpack embedded code
if os.path.exists('/content/code_bundle.tar.gz'):
    print("Extracting uploaded code_bundle.tar.gz...")
    !tar -xzf /content/code_bundle.tar.gz -C /content/code/
else:
    print("Self-extracting embedded pipeline source code...")
    b64_data = """..."""
    raw = base64.b64decode(b64_data)
    with tarfile.open(fileobj=io.BytesIO(raw), mode='r:gz') as tar:
        tar.extractall(path='/content/code')

print("Code successfully unpacked in /content/code/src:")
!ls -la /content/code/src

### Cell 4: Detect and Unpack Dataset
Detects any dataset archive or directory uploaded to `/content/` (e.g. `dataset.tar.gz`, `dataset.zip`, or `student_resource`).

In [ ]:
# 1. Download and extract dataset directly from Unstop CDN
import os

if not os.path.exists('/content/dataset/train/train_source1.tsv'):
    print('Downloading dataset directly from Unstop CDN...')
    !wget -q --show-progress -O /content/dataset.zip "https://cdn.unstop.com/files/6ab10eb3b23ba_student_resource.zip"
    print('Unpacking dataset...')
    !mkdir -p /content/temp_extract /content/dataset
    !unzip -q -o /content/dataset.zip -d /content/temp_extract/
    !if [ -d "/content/temp_extract/student_resource/dataset" ]; then \
        cp -r /content/temp_extract/student_resource/dataset/* /content/dataset/; \
     elif [ -d "/content/temp_extract/dataset" ]; then \
        cp -r /content/temp_extract/dataset/* /content/dataset/; \
     else \
        cp -r /content/temp_extract/* /content/dataset/; \
     fi
    !rm -rf /content/temp_extract /content/dataset.zip
else:
    print('Dataset already installed and ready in /content/dataset/')

print('\nDataset verification:')
!ls -lh /content/dataset/train /content/dataset/test


### Cell 5: Step 1 — Data Normalization (`prepare.py`)
Converts raw TSV files into normalized Parquet files with `anyascii` romanization, legal token mapping, and French `cedex` / `bp` address cleaning.

In [ ]:
%cd /content/code/src
!python prepare.py --data-dir /content/dataset --work-dir /content/work

### Cell 6: Step 2 — Candidate Blocking on Training Data (`run_blocking.py`)
Builds the inverted index and generates `train_cands.parquet` with the improved 98.2% recall blocker.

In [ ]:
%cd /content/code/src
!python run_blocking.py --work-dir /content/work --split train

### Cell 7: Step 3 — Leave-One-Country-Out (LOCO) Proxy Validation
Simulates an unseen country (like France in the test set) by training strictly on **US** and validating on **India**.
This outputs the real, labeled F0.5-vs-threshold curve for out-of-distribution transfer with **0 submissions burned**.

In [ ]:
%cd /content/code/src
!python train.py \
    --work-dir /content/work \
    --gt /content/dataset/train/train_ground_truth.tsv \
    --model-dir /content/work/models_loco \
    --holdout-country India

### Cell 8: Step 4 — Candidate Blocking on Test Data (`run_blocking.py`)
Generates `test_cands.parquet` using the exact same v3 blocker.

In [ ]:
%cd /content/code/src
!python run_blocking.py --work-dir /content/work --split test

### Cell 9: Step 5 — Full Model Training (v3)
Trains Stage 1 and Stage 2 LightGBM matchers on the complete training set with a standard 2% holdout.

In [ ]:
%cd /content/code/src
!python train.py \
    --work-dir /content/work \
    --gt /content/dataset/train/train_ground_truth.tsv \
    --model-dir /content/work/models_v3

### Cell 10: Step 6 — Test Inference & Decision Prediction (`predict.py`)
Produces `matching_results.tsv` and `candidate_pairs.tsv` in `/content/output/`.
*(If Cell 7 showed that an unseen country benefits from a specific threshold shift, you can pass `--country-thresholds`)*.

In [ ]:
%cd /content/code/src
!python predict.py \
    --work-dir /content/work \
    --model-dir /content/work/models_v3 \
    --out-dir /content/output

### Cell 11: Step 7 — Validate Formatting Before Submitting
Runs the official validation script. Must output `PASS`.

In [ ]:
!python /content/code/src/validate_submission.py \
    --matching /content/output/matching_results.tsv \
    --candidate /content/output/candidate_pairs.tsv \
    --test-dir /content/dataset/test

### Cell 12: Step 8 — Download Submission File to Your Mac
Downloads the validated `matching_results.tsv` ready for upload to Unstop as **Submission #2**.

In [ ]:
from google.colab import files
files.download('/content/output/matching_results.tsv')